# ECG Classification - PyTorch LSTM Model (v3-pytorch)**What's New in This Version:**- ✅ **PyTorch Implementation**: Converted from TensorFlow/Keras to PyTorch- ✅ **Fixed Data Leakage**: Scaler now fits ONLY on training data- ✅ **Stronger Regularization**: Increased dropout, added weight decay- ✅ **Direct ONNX Export**: Native PyTorch to ONNX conversion- ✅ **Manual Training Loop**: More control and transparency**Why This Fixes Overfitting:**1. **Data Leakage Fixed**: Previous version fit scaler on ALL data before splitting2. **Increased Dropout**: 0.3 → 0.5 (50% of neurons dropped during training)3. **Weight Decay**: L2 regularization added (weight_decay=1e-4)4. **Reduced Patience**: Early stopping patience 15 → 10 epochs5. **Better Validation**: Proper train/val/test isolation

## STEP 1: Import Libraries

In [ ]:
# Core librariesimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import (    confusion_matrix, ConfusionMatrixDisplay, classification_report,    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score)import warningswarnings.filterwarnings('ignore')# PyTorchimport torchimport torch.nn as nnimport torch.optim as optimfrom torch.utils.data import Dataset, DataLoader# Check PyTorch availabilityprint(f'PyTorch version: {torch.__version__}')print(f'CUDA available: {torch.cuda.is_available()}')device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Using device: {device}')# Set random seeds for reproducibilityRANDOM_STATE = 42torch.manual_seed(RANDOM_STATE)np.random.seed(RANDOM_STATE)if torch.cuda.is_available():    torch.cuda.manual_seed(RANDOM_STATE)

## STEP 2: Load ECG Dataset

In [ ]:
# Load the ECG dataset# Adjust path based on your environmentdf = pd.read_csv('../../dataset_aritmia_NEW.csv')print(f'Dataset Shape: {df.shape}')print(f'Columns: {df.columns.tolist()[:10]}... (showing first 10)')print(f'\nFirst few rows:')print(df.head())

## STEP 3: Exploratory Data Analysis

In [ ]:
# Check label distributionprint('Label Distribution:')print(df['label'].value_counts())print('\nPercentage:')print(df['label'].value_counts(normalize=True) * 100)# Visualize distributionfig, axes = plt.subplots(1, 2, figsize=(12, 4))df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])axes[0].set_title('Distribution of ECG Labels')axes[0].set_xlabel('Label (0=Normal, 1=Abnormal)')axes[0].set_ylabel('Count')axes[0].set_xticklabels(['Normal (0)', 'Abnormal (1)'], rotation=0)df['label'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['green', 'red'])axes[1].set_title('Label Proportion')axes[1].set_ylabel('')plt.tight_layout()plt.show()

## STEP 4: Data Preprocessing - **FIXING DATA LEAKAGE****CRITICAL CHANGE:**Previous version had data leakage because it fit the scaler on ALL data before splitting.This version fixes it by:1. Split data FIRST2. Fit scaler ONLY on training data3. Transform validation and test using training statistics

In [ ]:
# Separate features and labelsX = df.drop('label', axis=1).valuesy = df['label'].valuesprint(f'Features shape: {X.shape}')print(f'Labels shape: {y.shape}')# =============================================================================# STEP 4.1: SPLIT FIRST (before any preprocessing)# =============================================================================# This is the CORRECT order to prevent data leakage# First split: 80% train, 20% temp (for val + test)X_train, X_temp, y_train, y_temp = train_test_split(    X, y,    test_size=0.2,    random_state=RANDOM_STATE,    stratify=y)# Second split: 50% of temp = 10% validation, 10% testX_val, X_test, y_val, y_test = train_test_split(    X_temp, y_temp,    test_size=0.5,    random_state=RANDOM_STATE,    stratify=y_temp)print(f'\nSplit sizes:')print(f'Training:   {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)')print(f'Validation: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)')print(f'Test:       {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)')# =============================================================================# STEP 4.2: NORMALIZE - FIT ONLY ON TRAINING DATA# =============================================================================# This prevents information leakage from validation/test setsscaler = StandardScaler()X_train_normalized = scaler.fit_transform(X_train)  # Fit on training onlyX_val_normalized = scaler.transform(X_val)          # Transform using training statsX_test_normalized = scaler.transform(X_test)        # Transform using training statsprint(f'\nNormalization (training data only):')print(f'Mean: {scaler.mean_[:5]}...')print(f'Std: {scaler.scale_[:5]}...')# Reshape for LSTM (samples, timesteps, features)X_train_reshaped = X_train_normalized.reshape(-1, 188, 1)X_val_reshaped = X_val_normalized.reshape(-1, 188, 1)X_test_reshaped = X_test_normalized.reshape(-1, 188, 1)print(f'\nReshaped for LSTM:')print(f'Training: {X_train_reshaped.shape}')print(f'Validation: {X_val_reshaped.shape}')print(f'Test: {X_test_reshaped.shape}')

## STEP 5: PyTorch Dataset and DataLoader

In [ ]:
class ECGDataset(Dataset):    """Custom PyTorch Dataset for ECG data"""    def __init__(self, X, y):        self.X = torch.FloatTensor(X)        self.y = torch.LongTensor(y)        def __len__(self):        return len(self.X)        def __getitem__(self, idx):        return self.X[idx], self.y[idx]# Create datasetstrain_dataset = ECGDataset(X_train_reshaped, y_train)val_dataset = ECGDataset(X_val_reshaped, y_val)test_dataset = ECGDataset(X_test_reshaped, y_test)# Create dataloadersBATCH_SIZE = 32train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)print(f'DataLoaders created:')print(f'Training batches: {len(train_loader)}')print(f'Validation batches: {len(val_loader)}')print(f'Test batches: {len(test_loader)}')

## STEP 6: PyTorch LSTM Model Architecture**Regularization Improvements:**- Dropout increased from 0.3 to 0.5 (more aggressive)- Bidirectional LSTM with proper dropout- Batch normalization after LSTM layers

In [ ]:
class ECG_LSTM(nn.Module):    """    Bidirectional LSTM for ECG Classification        Architecture:    - 2 Bidirectional LSTM layers (64 and 32 units per direction)    - Dropout 0.5 (increased from 0.3 to prevent overfitting)    - Batch Normalization    - Dense layers for classification    """    def __init__(self, input_size=1, hidden_size=64, num_layers=2, num_classes=2, dropout=0.5):        super(ECG_LSTM, self).__init__()                # First Bidirectional LSTM layer        self.lstm1 = nn.LSTM(            input_size=input_size,            hidden_size=hidden_size,            num_layers=1,            batch_first=True,            bidirectional=True,            dropout=0        )        self.bn1 = nn.BatchNorm1d(hidden_size * 2)        self.dropout1 = nn.Dropout(dropout)                # Second Bidirectional LSTM layer        self.lstm2 = nn.LSTM(            input_size=hidden_size * 2,            hidden_size=hidden_size // 2,            num_layers=1,            batch_first=True,            bidirectional=True,            dropout=0        )        self.bn2 = nn.BatchNorm1d(hidden_size)        self.dropout2 = nn.Dropout(dropout)                # Fully connected layers        self.fc1 = nn.Linear(hidden_size, 64)        self.bn3 = nn.BatchNorm1d(64)        self.dropout3 = nn.Dropout(dropout)                self.fc2 = nn.Linear(64, 32)        self.dropout4 = nn.Dropout(dropout * 0.8)  # Slightly less dropout                self.fc3 = nn.Linear(32, num_classes)                self.relu = nn.ReLU()        def forward(self, x):        # x shape: (batch, seq_len, input_size)                # First LSTM layer        lstm_out, _ = self.lstm1(x)        # lstm_out shape: (batch, seq_len, hidden_size*2)                # BatchNorm requires (batch, features, seq_len)        lstm_out = lstm_out.permute(0, 2, 1)        lstm_out = self.bn1(lstm_out)        lstm_out = lstm_out.permute(0, 2, 1)        lstm_out = self.dropout1(lstm_out)                # Second LSTM layer        lstm_out, (hidden, _) = self.lstm2(lstm_out)        # Take the last hidden state from both directions        # hidden shape: (2, batch, hidden_size//2) for bidirectional        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)        # hidden shape: (batch, hidden_size)                hidden = self.bn2(hidden)        hidden = self.dropout2(hidden)                # Fully connected layers        out = self.fc1(hidden)        out = self.relu(out)        out = self.bn3(out)        out = self.dropout3(out)                out = self.fc2(out)        out = self.relu(out)        out = self.dropout4(out)                out = self.fc3(out)                return out# Create modelmodel = ECG_LSTM(    input_size=1,    hidden_size=64,    num_layers=2,    num_classes=2,    dropout=0.5).to(device)print('Model Architecture:')print(model)print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters())}')print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')

## STEP 7: Training Configuration and Loss Function**Key Changes from TensorFlow Version:**- Weight decay (L2 regularization) added: 1e-4- Dropout increased: 0.3 → 0.5- Early stopping patience reduced: 15 → 10- Class weights for imbalanced data

In [ ]:
# Compute class weights for imbalanced datafrom sklearn.utils.class_weight import compute_class_weightclass_weights = compute_class_weight(    class_weight='balanced',    classes=np.unique(y_train),    y=y_train)class_weights = torch.FloatTensor(class_weights).to(device)print(f'Class weights: {class_weights}')# Loss function with class weightscriterion = nn.CrossEntropyLoss(weight=class_weights)# Optimizer with weight decay (L2 regularization)optimizer = optim.Adam(    model.parameters(),    lr=0.001,    weight_decay=1e-4  # L2 regularization)# Learning rate schedulerscheduler = optim.lr_scheduler.ReduceLROnPlateau(    optimizer,    mode='min',    factor=0.5,    patience=5,    min_lr=1e-6,    verbose=True)print('\nTraining Configuration:')print(f'Loss function: CrossEntropyLoss with class weights')print(f'Optimizer: Adam (lr=0.001, weight_decay=1e-4)')print(f'Scheduler: ReduceLROnPlateau (factor=0.5, patience=5)')

## STEP 8: Training and Validation Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):    """Train for one epoch"""    model.train()    running_loss = 0.0    correct = 0    total = 0        for inputs, labels in loader:        inputs, labels = inputs.to(device), labels.to(device)                # Forward pass        optimizer.zero_grad()        outputs = model(inputs)        loss = criterion(outputs, labels)                # Backward pass        loss.backward()        optimizer.step()                # Statistics        running_loss += loss.item()        _, predicted = outputs.max(1)        total += labels.size(0)        correct += predicted.eq(labels).sum().item()        epoch_loss = running_loss / len(loader)    epoch_acc = correct / total    return epoch_loss, epoch_accdef validate_epoch(model, loader, criterion, device):    """Validate for one epoch"""    model.eval()    running_loss = 0.0    correct = 0    total = 0        with torch.no_grad():        for inputs, labels in loader:            inputs, labels = inputs.to(device), labels.to(device)                        # Forward pass            outputs = model(inputs)            loss = criterion(outputs, labels)                        # Statistics            running_loss += loss.item()            _, predicted = outputs.max(1)            total += labels.size(0)            correct += predicted.eq(labels).sum().item()        epoch_loss = running_loss / len(loader)    epoch_acc = correct / total    return epoch_loss, epoch_accprint('Training and validation functions defined.')

## STEP 9: Training Loop with Early Stopping**Improvements:**- Manual early stopping (patience=10, reduced from 15)- Track best model weights- Learning rate scheduling- Progress monitoring

In [ ]:
# Training parametersNUM_EPOCHS = 100PATIENCE = 10  # Reduced from 15best_val_loss = float('inf')patience_counter = 0train_losses = []val_losses = []train_accs = []val_accs = []print('Starting Training...')print('=' * 70)for epoch in range(NUM_EPOCHS):    # Train    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)        # Validate    val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)        # Store metrics    train_losses.append(train_loss)    val_losses.append(val_loss)    train_accs.append(train_acc)    val_accs.append(val_acc)        # Learning rate scheduling    scheduler.step(val_loss)    current_lr = optimizer.param_groups[0]['lr']        # Print progress    if (epoch + 1) % 5 == 0 or epoch < 5:        print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] '              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | '              f'LR: {current_lr:.6f}')        # Early stopping    if val_loss < best_val_loss:        best_val_loss = val_loss        patience_counter = 0        # Save best model        torch.save({            'epoch': epoch,            'model_state_dict': model.state_dict(),            'optimizer_state_dict': optimizer.state_dict(),            'val_loss': val_loss,            'val_acc': val_acc,        }, 'ecg_lstm_pytorch_best.pth')        if (epoch + 1) % 5 == 0:            print(f'  → New best model saved (val_loss: {val_loss:.4f})')    else:        patience_counter += 1        if patience_counter >= PATIENCE:            print(f'\nEarly stopping triggered at epoch {epoch+1}')            breakprint('\nTraining Complete!')print('=' * 70)# Load best modelcheckpoint = torch.load('ecg_lstm_pytorch_best.pth')model.load_state_dict(checkpoint['model_state_dict'])print(f'Best model loaded from epoch {checkpoint["epoch"]+1}')print(f'Best validation loss: {checkpoint["val_loss"]:.4f}')print(f'Best validation accuracy: {checkpoint["val_acc"]:.4f}')

## STEP 10: Visualize Training History

In [ ]:
# Plot training historyfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))# Loss plotax1.plot(train_losses, label='Training Loss', linewidth=2)ax1.plot(val_losses, label='Validation Loss', linewidth=2)ax1.set_xlabel('Epoch')ax1.set_ylabel('Loss')ax1.set_title('Training and Validation Loss')ax1.legend()ax1.grid(True)# Accuracy plotax2.plot(train_accs, label='Training Accuracy', linewidth=2)ax2.plot(val_accs, label='Validation Accuracy', linewidth=2)ax2.set_xlabel('Epoch')ax2.set_ylabel('Accuracy')ax2.set_title('Training and Validation Accuracy')ax2.legend()ax2.grid(True)plt.tight_layout()plt.show()print(f'\nFinal Metrics (Best Model):')print(f'Training Accuracy: {train_accs[checkpoint["epoch"]]:.4f}')print(f'Validation Accuracy: {checkpoint["val_acc"]:.4f}')print(f'\nExpected: 82-92% (realistic after fixing data leakage)')

## STEP 11: Comprehensive Evaluation on Test Set

In [ ]:
# Evaluate on test settest_loss, test_acc = validate_epoch(model, test_loader, criterion, device)print('Test Set Evaluation:')print('=' * 70)print(f'Test Loss: {test_loss:.4f}')print(f'Test Accuracy: {test_acc:.4f}')# Get predictions for detailed metricsmodel.eval()all_preds = []all_labels = []with torch.no_grad():    for inputs, labels in test_loader:        inputs = inputs.to(device)        outputs = model(inputs)        _, predicted = outputs.max(1)        all_preds.extend(predicted.cpu().numpy())        all_labels.extend(labels.numpy())all_preds = np.array(all_preds)all_labels = np.array(all_labels)# Classification reportfrom sklearn.metrics import classification_report, confusion_matrixprint('\nClassification Report:')print(classification_report(all_labels, all_preds,                           target_names=['Normal (0)', 'Abnormal (1)']))# Confusion matrixcm = confusion_matrix(all_labels, all_preds)print('\nConfusion Matrix:')print(cm)# Visualize confusion matrixfig, ax = plt.subplots(figsize=(8, 6))im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)ax.figure.colorbar(im, ax=ax)ax.set(xticks=np.arange(cm.shape[1]),       yticks=np.arange(cm.shape[0]),       xticklabels=['Normal', 'Abnormal'],       yticklabels=['Normal', 'Abnormal'],       title='Confusion Matrix',       ylabel='True label',       xlabel='Predicted label')# Add text annotationsthresh = cm.max() / 2.for i in range(cm.shape[0]):    for j in range(cm.shape[1]):        ax.text(j, i, format(cm[i, j], 'd'),                ha="center", va="center",                color="white" if cm[i, j] > thresh else "black")plt.tight_layout()plt.show()

## STEP 12: Save PyTorch Model and Scaler

In [ ]:
# Save final PyTorch modeltorch.save({    'model_state_dict': model.state_dict(),    'model_config': {        'input_size': 1,        'hidden_size': 64,        'num_layers': 2,        'num_classes': 2,        'dropout': 0.5    },    'test_acc': test_acc,    'test_loss': test_loss}, 'ecg_lstm_v3_pytorch_final.pth')print('PyTorch model saved: ecg_lstm_v3_pytorch_final.pth')# Save scaler (same as before, for consistency)import joblibjoblib.dump(scaler, 'scaler_v3_pytorch.pkl')print('Scaler saved: scaler_v3_pytorch.pkl')print('\nNote: Scaler fits ONLY on training data (data leakage fixed!)')

## STEP 13: Export to ONNX Format**Benefits of PyTorch → ONNX:**- Direct export (no tf2onnx needed)- Cleaner conversion process- Better compatibility- Lightweight deployment

In [ ]:
print('Exporting to ONNX format...')print('=' * 70)try:    # Set model to evaluation mode    model.eval()        # Create dummy input (batch_size=1, seq_len=188, features=1)    dummy_input = torch.randn(1, 188, 1).to(device)        # Export to ONNX    onnx_path = 'ecg_lstm_v3_pytorch_final.onnx'    torch.onnx.export(        model,        dummy_input,        onnx_path,        export_params=True,        opset_version=13,        do_constant_folding=True,        input_names=['input'],        output_names=['output'],        dynamic_axes={            'input': {0: 'batch_size'},            'output': {0: 'batch_size'}        }    )        print(f'✓ ONNX model exported: {onnx_path}')        # Verify ONNX model    import onnxruntime as ort    import onnx        # Check model    onnx_model = onnx.load(onnx_path)    onnx.checker.check_model(onnx_model)    print('✓ ONNX model is valid')        # Test inference    ort_session = ort.InferenceSession(onnx_path)        # Get input/output names    input_name = ort_session.get_inputs()[0].name    output_name = ort_session.get_outputs()[0].name        print(f'✓ ONNX model verified')    print(f'  Input: {input_name}, shape: {ort_session.get_inputs()[0].shape}')    print(f'  Output: {output_name}, shape: {ort_session.get_outputs()[0].shape}')        # Test with sample data    test_input = np.random.randn(1, 188, 1).astype(np.float32)    ort_output = ort_session.run([output_name], {input_name: test_input})    print(f'✓ Test inference successful, output shape: {ort_output[0].shape}')        print('\n' + '=' * 70)    print('EXPORT SUMMARY:')    print('=' * 70)    print(f'PyTorch model:  ecg_lstm_v3_pytorch_final.pth')    print(f'ONNX model:     {onnx_path}')    print(f'Scaler:         scaler_v3_pytorch.pkl')    print('\nFor deployment:')    print('  pip install onnxruntime numpy pandas')    print('  # Use ONNX model with realtime_frontend.py')    print('  # No PyTorch/TensorFlow needed!')    except Exception as e:    print(f'✗ ONNX export failed: {e}')    print('  PyTorch model is saved and can be exported manually.')    import traceback    traceback.print_exc()

## STEP 14: Summary### Key Improvements in This PyTorch Version:1. **Data Leakage Fixed** ✅   - Scaler now fits ONLY on training data   - Proper train/val/test isolation   - Realistic accuracy metrics (82-92% expected)2. **Stronger Regularization** ✅   - Dropout: 0.3 → 0.5 (67% increase)   - Weight decay: 1e-4 (L2 regularization)   - Early stopping patience: 15 → 103. **PyTorch Benefits** ✅   - More control over training process   - Direct ONNX export (no tf2onnx)   - Better for research and experimentation   - Clearer code structure4. **Production Ready** ✅   - ONNX model for lightweight deployment   - Compatible with realtime_frontend.py   - Cross-platform (Windows/Linux/macOS)### Expected Performance:- **Before fix**: 99-100% (artificial due to data leakage)- **After fix**: 82-92% (realistic model performance)### Next Steps:1. Use ONNX model for deployment2. Apply same fixes to v2 (CNN) and v5 (Transformer)3. Compare performance across all three architectures